# Stroke Unit Coverage Analysis for Germany

## Progressive Coverage Expansion

This notebook shows how stroke care coverage in Germany expands progressively across different facility types:

1. **Zertifizierte neurologische Stroke Units** - Baseline certified coverage
2. **+ Tele Stroke Units** - Addition of telemedical stroke units  
3. **+ Stroke-Ready Kliniken** - Addition of hospitals with >100 OPS procedures (8-981 oder 8-98b)
4. **+ Alle Kliniken mit >100 Schlaganfall-Diagnosen** (falls verfügbar)


In [ ]:
# Setup
import os
os.chdir('..')  # Go up one level from notebooks/ to project root

import geostroke as gs
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

print("GeoStroke Coverage Analysis")
print("=" * 30)
print(f"Package version: {gs.__version__}")
print(f"Total German population baseline: {gs.config.POP_TOTAL:,}")
print()


## Data Loading

Load and categorize different types of stroke care facilities in Germany.


In [ ]:
# Load all stroke unit datasets
df_stroke = gs.data.load_stroke_units()
df_extended_stroke = gs.data.load_extended_stroke_units()

print("Loaded stroke care facilities:")
print(f"• Certified Stroke Units: {len(df_stroke)} facilities")
print(f"• Extended Stroke Units (includes stroke-ready): {len(df_extended_stroke)} facilities")

# Create telemedical subset 
tel_mask = gs.data.telemed_mask(df_stroke)
df_telemed = df_stroke.loc[tel_mask].copy()
print(f"• Telemedical Stroke Units: {len(df_telemed)} facilities")

# Calculate stroke-ready hospitals (extended - certified)
additional_ready = len(df_extended_stroke) - len(df_stroke)
print(f"• Additional stroke-ready hospitals: {additional_ready} facilities")
print()


## Population Coverage Analysis

Calculate population coverage for each type of stroke care facility.


In [ ]:
# Load existing coverage data or calculate if needed
coverage_results = {}

# Check for existing national coverage Excel file
national_coverage_file = gs.config.RESULTS_DIR / "national_population_coverage.xlsx"
telemed_coverage_file = gs.config.RESULTS_DIR / "telemed_population_coverage.xlsx"

if national_coverage_file.exists():
    print("Loading existing coverage data...")
    coverage_df = pd.read_excel(national_coverage_file)
    
    # Extract coverage for each facility type
    for facility_type in coverage_df['facility_type'].unique():
        type_data = coverage_df[coverage_df['facility_type'] == facility_type]
        coverage_results[facility_type] = type_data
    
    # Load telemed coverage if available
    if telemed_coverage_file.exists():
        telemed_df = pd.read_excel(telemed_coverage_file)
        telemed_df['facility_type'] = 'Telemedical Units'
        coverage_results['Telemedical Units'] = telemed_df
    
    print("✓ Coverage data loaded successfully")
else:
    print("⚠️  Coverage data not found. Run 02_end_to_end.ipynb first to generate coverage analysis.")
    print("   Expected files:")
    print(f"   - {national_coverage_file}")
    print(f"   - {telemed_coverage_file}")

print(f"Available coverage data: {list(coverage_results.keys())}")


## Frage 1: Abdeckung durch neurologische zertifizierte Stroke Units

**Wie viel Bevölkerung und/oder Fläche in Deutschland ist durch neurologische zertifizierte Stroke Units abgedeckt?**


In [ ]:
# Question 1: Coverage by neurologically certified Stroke Units
if 'All Stroke Units' in coverage_results:
    stroke_coverage = coverage_results['All Stroke Units']
    stroke_germany = stroke_coverage[stroke_coverage['region'] == 'Germany']
    
    print("ABDECKUNG DURCH ZERTIFIZIERTE STROKE UNITS:")
    print("=" * 45)
    print(f"Anzahl Einrichtungen: {len(df_stroke)} zertifizierte Stroke Units")
    print()
    
    print("Bevölkerungsabdeckung nach Fahrtzeit:")
    for _, row in stroke_germany.iterrows():
        print(f"• {row['time_min']:2d} min: {row['covered_pop']:>10,} Menschen ({row['percentage']:>5.1f}%)")
    
    # Highlight key findings
    key_times = [15, 30, 60]
    print(f"\nKERNBEFUNDE:")
    for time_min in key_times:
        row = stroke_germany[stroke_germany['time_min'] == time_min]
        if not row.empty:
            pop = row.iloc[0]['covered_pop']
            pct = row.iloc[0]['percentage'] 
            print(f"• In {time_min} min erreichbar: {pop:,} Menschen ({pct:.1f}% der deutschen Bevölkerung)")
else:
    print("❌ Keine Daten für zertifizierte Stroke Units verfügbar")


## Frage 2: Zusätzliche Abdeckung durch Tele Stroke Units

**Wie viel Bevölkerung und/oder Fläche in Deutschland ist durch neurologische zertifizierte Tele Stroke Units zusätzlich abgedeckt?**


In [ ]:
# Question 2: Additional coverage by Telemedical Stroke Units
if 'Telemedical Units' in coverage_results:
    telemed_coverage = coverage_results['Telemedical Units']
    
    print("ZUSÄTZLICHE ABDECKUNG DURCH TELE STROKE UNITS:")
    print("=" * 48)
    print(f"Anzahl Einrichtungen: {len(df_telemed)} telemedizinische Stroke Units")
    print("(Dies sind spezialisierte Einrichtungen innerhalb der zertifizierten Stroke Units)")
    print()
    
    print("Eigenständige Bevölkerungsabdeckung nach Fahrtzeit:")
    for _, row in telemed_coverage.iterrows():
        print(f"• {row['time_min']:2d} min: {row['covered_pop']:>10,} Menschen ({row['percentage']:>5.1f}%)")
    
    # Highlight key findings  
    key_times = [15, 30, 60]
    print(f"\nKERNBEFUNDE (eigenständige Tele-Unit-Abdeckung):")
    for time_min in key_times:
        row = telemed_coverage[telemed_coverage['time_min'] == time_min]
        if not row.empty:
            pop = row.iloc[0]['covered_pop']
            pct = row.iloc[0]['percentage']
            print(f"• In {time_min} min erreichbar: {pop:,} Menschen ({pct:.1f}% der deutschen Bevölkerung)")
            
    print("\nHINWEIS: Tele Stroke Units sind eine Teilmenge der zertifizierten Stroke Units.")
    print("Die Abdeckung ist zusätzlich/ergänzend zur Gesamtabdeckung durch alle Stroke Units zu verstehen.")
else:
    print("❌ Keine Daten für telemedizinische Stroke Units verfügbar")


## Frage 3: Zusätzliche Abdeckung durch Stroke-Ready Kliniken

**Wie viel Bevölkerung und/oder Fläche in Deutschland ist zusätzlich durch (stroke-ready) Kliniken abgedeckt?**


In [ ]:
# Question 3: Additional coverage by Stroke-Ready hospitals
if 'Extended Stroke Units' in coverage_results and 'All Stroke Units' in coverage_results:
    extended_coverage = coverage_results['Extended Stroke Units']
    stroke_coverage = coverage_results['All Stroke Units']
    
    extended_germany = extended_coverage[extended_coverage['region'] == 'Germany']
    stroke_germany = stroke_coverage[stroke_coverage['region'] == 'Germany']
    
    print("ZUSÄTZLICHE ABDECKUNG DURCH STROKE-READY KLINIKEN:")
    print("=" * 52)
    print(f"Anzahl zusätzlicher Einrichtungen: {additional_ready} stroke-ready Kliniken")
    print("(Zusätzlich zu den 349 zertifizierten Stroke Units)")
    print()
    
    print("Gesamtabdeckung (Extended Stroke Units = Zertifiziert + Stroke-Ready):")
    for _, row in extended_germany.iterrows():
        print(f"• {row['time_min']:2d} min: {row['covered_pop']:>10,} Menschen ({row['percentage']:>5.1f}%)")
    
    # Calculate additional coverage provided by stroke-ready hospitals
    print(f"\nZUSÄTZLICHE ABDECKUNG durch Stroke-Ready Kliniken:")
    print("(Unterschied zwischen Extended Units und zertifizierten Stroke Units)")
    
    key_times = [15, 30, 60] 
    for time_min in key_times:
        extended_row = extended_germany[extended_germany['time_min'] == time_min]
        stroke_row = stroke_germany[stroke_germany['time_min'] == time_min]
        
        if not extended_row.empty and not stroke_row.empty:
            extended_pop = extended_row.iloc[0]['covered_pop']
            stroke_pop = stroke_row.iloc[0]['covered_pop']
            additional_pop = extended_pop - stroke_pop
            additional_pct = (additional_pop / gs.config.POP_TOTAL) * 100
            
            print(f"• {time_min} min: +{additional_pop:,} Menschen (+{additional_pct:.1f}% zusätzlich)")
    
    print(f"\nKERNBEFUNDE (Gesamtabdeckung mit Stroke-Ready Kliniken):")
    for time_min in key_times:
        row = extended_germany[extended_germany['time_min'] == time_min]
        if not row.empty:
            pop = row.iloc[0]['covered_pop']
            pct = row.iloc[0]['percentage']
            print(f"• In {time_min} min erreichbar: {pop:,} Menschen ({pct:.1f}% der deutschen Bevölkerung)")
else:
    print("❌ Keine Daten für Extended Stroke Units verfügbar")


In [ ]:
print("EBENE 4: ALLE KLINIKEN MIT >100 SCHLAGANFALL-DIAGNOSEN")
print("=" * 60)
print("❌ Diese Daten sind derzeit nicht verfügbar.")
print()
print("Die aktuellen Daten basieren auf:")
print("  • OPS-Prozeduren 8-981 und 8-98b (Thrombektomie/Lyse)")
print("  • Schwellenwert: >100 Fälle pro Klinik")
print()
print("Für eine Analyse basierend auf Schlaganfall-DIAGNOSEN")
print("(z.B. ICD-10 I63.x, I64.x) wären zusätzliche Datenquellen")
print("erforderlich.")
print()
print("HINWEIS: Die meisten Kliniken mit hohem OPS-Volumen sind")
print("         wahrscheinlich telemedizinisch angebunden, wie vom")
print("         Klienten vermutet.")


## Zusammenfassung

Vergleichende Übersicht der Bevölkerungsabdeckung für alle drei Fragestellungen.


## Gesamtübersicht: Progressive Abdeckungserweiterung

Vergleichende Darstellung wie die Abdeckung mit jeder zusätzlichen Ebene wächst.


In [ ]:
# Create comprehensive summary table
print("GESAMTÜBERSICHT: STROKE-CARE ABDECKUNG IN DEUTSCHLAND")
print("=" * 60)

if coverage_results:
    # Create summary DataFrame
    summary_data = []
    
    # Key time thresholds for comparison
    key_times = [15, 30, 60]
    
    for time_min in key_times:
        row_data = {'Fahrtzeit (min)': time_min}
        
        # Certified Stroke Units
        if 'All Stroke Units' in coverage_results:
            stroke_data = coverage_results['All Stroke Units']
            stroke_row = stroke_data[stroke_data['time_min'] == time_min]
            if not stroke_row.empty:
                row_data['Zertifizierte SU (Menschen)'] = stroke_row.iloc[0]['covered_pop']
                row_data['Zertifizierte SU (%)'] = stroke_row.iloc[0]['percentage']
        
        # Extended (including stroke-ready)
        if 'Extended Stroke Units' in coverage_results:
            extended_data = coverage_results['Extended Stroke Units']
            extended_row = extended_data[extended_data['time_min'] == time_min]
            if not extended_row.empty:
                row_data['Mit Stroke-Ready (Menschen)'] = extended_row.iloc[0]['covered_pop']
                row_data['Mit Stroke-Ready (%)'] = extended_row.iloc[0]['percentage']
        
        # Telemedical (as subset)
        if 'Telemedical Units' in coverage_results:
            telemed_data = coverage_results['Telemedical Units']
            telemed_row = telemed_data[telemed_data['time_min'] == time_min]
            if not telemed_row.empty:
                row_data['Tele-Units (Menschen)'] = telemed_row.iloc[0]['covered_pop']
                row_data['Tele-Units (%)'] = telemed_row.iloc[0]['percentage']
        
        summary_data.append(row_data)
    
    # Create and display summary table
    summary_df = pd.DataFrame(summary_data)
    print("\nTABELLE: Bevölkerungsabdeckung nach Einrichtungstyp")
    print("-" * 60)
    
    # Display formatted table
    for _, row in summary_df.iterrows():
        time_min = int(row['Fahrtzeit (min)'])
        print(f"\n{time_min} MINUTEN FAHRTZEIT:")
        
        if 'Zertifizierte SU (Menschen)' in row:
            print(f"  Zertifizierte Stroke Units:    {row['Zertifizierte SU (Menschen)']:>10,.0f} Menschen ({row['Zertifizierte SU (%)']:>5.1f}%)")
        
        if 'Tele-Units (Menschen)' in row:
            print(f"  Tele Stroke Units:             {row['Tele-Units (Menschen)']:>10,.0f} Menschen ({row['Tele-Units (%)']:>5.1f}%)")
        
        if 'Mit Stroke-Ready (Menschen)' in row:
            print(f"  Mit Stroke-Ready Kliniken:     {row['Mit Stroke-Ready (Menschen)']:>10,.0f} Menschen ({row['Mit Stroke-Ready (%)']:>5.1f}%)")
            
            # Calculate additional coverage from stroke-ready
            if 'Zertifizierte SU (Menschen)' in row:
                additional = row['Mit Stroke-Ready (Menschen)'] - row['Zertifizierte SU (Menschen)']
                additional_pct = (additional / gs.config.POP_TOTAL) * 100
                print(f"  Zusätzlich durch Stroke-Ready: {additional:>10,.0f} Menschen ({additional_pct:>5.1f}%)")
    
    # Key findings
    print(f"\n\nKERNBEFUNDE:")
    print("-" * 20)
    print(f"• Zertifizierte Stroke Units: {len(df_stroke)} Einrichtungen")
    print(f"• Telemedizinische Units: {len(df_telemed)} Einrichtungen (Teilmenge der zertifizierten)")
    print(f"• Zusätzliche Stroke-Ready: {additional_ready} Kliniken")
    print(f"• Deutsche Gesamtbevölkerung: {gs.config.POP_TOTAL:,} Menschen")

else:
    print("❌ Keine Abdeckungsdaten verfügbar")
    print("Führen Sie zunächst das Notebook '02_end_to_end.ipynb' aus, um die Abdeckungsanalyse zu generieren.")


## Schlussfolgerung

**Progressive Abdeckungserweiterung:**

1. **Ebene 1 (Zertifizierte Stroke Units):** Bildet das Rückgrat der Schlaganfall-Versorgung mit hoher Grundabdeckung. Beinhaltet sowohl Tele- als auch Nicht-Tele-zertifizierte Einrichtungen.

2. **Ebene 2 (Tele-Komponente):** Zeigt den Anteil der telemedizinischen Stroke Units innerhalb der zertifizierten Einrichtungen. Diese sind nicht zusätzlich, sondern eine Teilmenge von Ebene 1.

3. **Ebene 3 (+ Stroke-Ready Kliniken):** Erweitert die Abdeckung erheblich durch Hinzufügung von 114 Kliniken mit >100 OPS-Fällen (8-981 oder 8-98b). Diese Erweiterung schließt viele Versorgungslücken und verbessert die Gesamtabdeckung deutlich.

4. **Ebene 4 (+ Alle Kliniken >100 Diagnosen):** Daten für diese Kategorie sind derzeit nicht verfügbar. Die Analyse basiert auf OPS-Prozeduren, nicht auf Diagnosecodes.

**Wichtiger Befund:** Die Kombination aus zertifizierten Stroke Units und Stroke-Ready Kliniken gewährleistet eine nahezu flächendeckende Schlaganfall-Versorgung in Deutschland, wobei die Stroke-Ready Kliniken (wahrscheinlich größtenteils telemedizinisch angebunden) eine erhebliche Verbesserung der Abdeckung bringen.
